In [1]:

import pandas as pd
from ytmusicapi import YTMusic
cache = {}

from fuzzywuzzy import fuzz
from fuzzywuzzy import process

yt = YTMusic('../headers_auth.json')

# Get a set of videoIds for thumbs up and thumbs down
def get_playlist_track_ids(playlist_id, print_stuff=False):
    playlist_meta = yt.get_playlist(playlist_id, limit=10000)
    track_ids = set()
    for i, track in enumerate(playlist_meta['tracks']):
        if print_stuff:
            try:
                print('(%d/%d) %s - %s - %s' % (
                    i+1, len(playlist_meta['tracks']), track['artists'][0]['name'], track['album']['name'], track['title']))
            except Exception as e:
                print(e)
        track_ids.add(track['videoId'])
    return track_ids



In [2]:
db = pd.read_csv('../../music-sources-unified/ytmusic_all_database.tsv', sep='\t', index_col=0)
db.columns

Index(['title', 'album', 'likeStatus', 'duration', 'artistId', 'artist',
       'albumId', 'playlists', 'keywords', 'averageRating', 'viewCount',
       'release', 'isAvailable', 'isExplicit', 'albumArtist',
       'albumTrackCount', 'albumDuration', 'albumYear', 'albumType',
       'fuzzy_id', 'fuzzy_album_id'],
      dtype='object')

In [61]:
db['entry_key'] = db['artist'].str.lower() + ' - ' +  db['title'].str.lower() 
db['artist_cat'] = db['artist'].str.slice(0,4).str.lower()

MATCH_SCORE_THRESH=95
added_ids = set()
for name, cat in db.groupby('artist_cat'):
    search_pool = cat.loc[~cat.index.isin(added_ids)]
    for e in cat.itertuples():
        if e.likeStatus == 'LIKE':
            exact_matches = cat.loc[cat.entry_key == e.entry_key]
            if len(exact_matches.loc[exact_matches.likeStatus != 'LIKE']):
                if len(exact_matches.albumId.unique()) > 1:
                    added_ids.add(e.Index)
                    liked=exact_matches.loc[exact_matches.likeStatus == 'LIKE']
                    added_ids.add(liked.sort_values('viewCount', ascending=False).index[0])
                    notliked=exact_matches.loc[exact_matches.likeStatus != 'LIKE']
                    added_ids.update(notliked.index.unique())

                else:
                #     # print('exact matches are from same album...', exact_matches.album.unique())
                    continue

            dont_search_ids = frozenset([e.Index]) | added_ids
            search_pool = cat.loc[~cat.index.isin(dont_search_ids)]
            if not len(search_pool):
                continue
            # print(e.entry_key, len(search_pool.entry_key.values))
            res = process.extract(e.entry_key, search_pool.entry_key.values, limit=5, scorer=fuzz.ratio)
            for r_key, score in res:
                if score > MATCH_SCORE_THRESH:
                    matches = search_pool.loc[search_pool.entry_key==r_key]
                    if len(matches.loc[matches.likeStatus != 'LIKE']):
                        if len(matches.albumId.unique()) > 1:
                            added_ids.add(e.Index)
                            added_ids.update(matches.index.unique())
                            # print(score,'\t', e.entry_key, '-->', r_key)
                        else:
                            continue


print(len(added_ids))

3031


6420
6385
2167
3031

In [57]:
6420

6420


In [20]:
search_pool

,title,album,likeStatus,duration,artistId,artist,albumId,playlists,keywords,averageRating,...,isExplicit,albumArtist,albumTrackCount,albumDuration,albumYear,albumType,fuzzy_id,fuzzy_album_id,entry_key,artist_cat
9LLuMxsOeoE,Yadnus,Myth Takes,INDIFFERENT,5:14,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_2LyolNjViS6,['NYC Indie Dance'],"['!!!', 'Myth Takes', 'Yadnus']",4.553551,...,NaN,!!!,10.0,48 minutes,2007.0,Album,!!! myth takes yadnus,!!! - myth takes,!!! - yadnus,!!!
m-5hskv-c84,Yadnus,Myth Takes,INDIFFERENT,5:14,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_2LyolNjViS6,['NYC Indie Dance'],"['!!!', 'Myth Takes', 'Yadnus']",5.000000,...,NaN,!!!,10.0,48 minutes,2007.0,Album,!!! myth takes yadnus,!!! - myth takes,!!! - yadnus,!!!
wVKdoRRjoCM,One Girl / One Boy,One Girl / One Boy,LIKE,4:04,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_GGnfq91bYrf,"['Your Likes', 'x_r.futurefunkairlines_tracks_...",NaN,NaN,...,False,!!!,1.0,"4 minutes, 4 seconds",2013.0,Single,!!! one girl / one boy one girl / one boy,!!! - one girl / one boy,!!! - one girl / one boy,!!!
xOk88bbxzPQ,"Jamie, My Intentions Are Bass","Strange Weather, Isn't It?",INDIFFERENT,5:07,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,['NYC Indie Dance'],"['!!! Strange Weather', ""Isn't It? Jamie"", 'My...",4.450135,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? jamie, my inten...","!!! - strange weather, isn't it?","!!! - jamie, my intentions are bass",!!!
tA1VaVj1kZQ,AM/FM,"Strange Weather, Isn't It?",INDIFFERENT,4:56,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? AM/FM""]",4.296703,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? am/fm","!!! - strange weather, isn't it?",!!! - am/fm,!!!
LoAiuQJ0qpM,Even Judas Gave Jesus A Kiss,"Strange Weather, Isn't It?",INDIFFERENT,5:46,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? Even Judas ...",3.697675,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? even judas gave...","!!! - strange weather, isn't it?",!!! - even judas gave jesus a kiss,!!!
aCqM0ZCu1ys,Wannagain Wannagain,"Strange Weather, Isn't It?",INDIFFERENT,3:50,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? Wannagain W...",4.111111,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? wannagain wanna...","!!! - strange weather, isn't it?",!!! - wannagain wannagain,!!!
bBQgf3cKkEA,Steady As The Sidewalk Cracks,"Strange Weather, Isn't It?",INDIFFERENT,4:26,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? Steady As T...",4.101604,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? steady as the s...","!!! - strange weather, isn't it?",!!! - steady as the sidewalk cracks,!!!
pAu6DGOefQg,Hollow,"Strange Weather, Isn't It?",INDIFFERENT,2:49,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? Hollow""]",3.666667,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? hollow","!!! - strange weather, isn't it?",!!! - hollow,!!!
3oUJM2i6m64,The Most Certain Sure,"Strange Weather, Isn't It?",INDIFFERENT,5:40,UC_kiReqJMAcQ9F9gf10QMTg,!!!,MPREb_9B1RPFLWvbm,[nan],"['!!! Strange Weather', ""Isn't It? The Most Ce...",4.373134,...,NaN,!!!,9.0,40 minutes,2010.0,Album,"!!! strange weather, isn't it? the most certai...","!!! - strange weather, isn't it?",!!! - the most certain sure,!!!


In [4]:
print(e.Index)

9LLuMxsOeoE


In [7]:
playlists = pd.DataFrame(yt.get_library_playlists(limit=500))
jb = playlists.loc[playlists.title.str.contains('jukebox_19')]
jb

,title,playlistId,thumbnails,count
197,jukebox_1960s,PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs,[{'url': 'https://lh3.googleusercontent.com/mI...,695
198,jukebox_1950s,PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj,[{'url': 'https://lh3.googleusercontent.com/FL...,772


jukebox_1960s PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs
jukebox_1960s
jukebox_1950s PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj
jukebox_1950s


In [124]:
# %%time
for i, row in jb.iterrows():
    print(row.title, row.playlistId)
    pl_name = row.title
    id = row.playlistId
    new_pl_name = pl_name + '_new'
    pl = yt.get_playlist(id, limit=1000)

    new_pl_ids = []
    for tk in pl['tracks']:        
        query = f"{tk['artists'][0]['name']} {tk['title']}"
        if query not in cache:
            cache[query] = yt.search(query, filter='songs')
        res = cache[query]
        TOP_N = 6
        choices = []
        # take first result, or any other results if they are in db
        for r in res[0:TOP_N]:
            if r['videoId'] in db.index:
                if vid not in cache:  
                    cache[vid]=yt.get_song(vid)
                if cache[vid]['playabilityStatus']['status'] == 'OK':            
                    choices.append((r['videoId'], db.loc[r['videoId']]['likeStatus']))
                    continue
                else:
                    print(f"db entry videoId no longer found!:\n{db.loc[r['videoId']]}")
            if not len(choices):
                choices.append((r['videoId'], None))
        
        if len(choices) == 1:
            choices_filt = choices
        else:
            rating ={'LIKE': [], 'DISLIKE': [], 'INDIFFERENT': [], 'NONE': []}
            for c in choices:
                vid, likeStatus = c                
                rating[str(likeStatus).upper()].append(c)
            choices_filt = []
            if len(rating['LIKE']):
                choices_filt = rating['LIKE']
            if not len(choices_filt)and len(rating['DISLIKE']):
                choices_filt = [rating['DISLIKE'][0]]
            if not len(choices_filt) and len(rating['INDIFFERENT']):
                choices_filt = [rating['INDIFFERENT'][0]]
            if not len(choices_filt):
                choices_filt = [rating['NONE'][0]]
        for c in choices_filt:
            vid, likeStatus = c                
            new_pl_ids.append(vid)
            if tk['likeStatus'] !='INDIFFERENT' and tk['likeStatus'] != likeStatus:
                yt.rate_song(vid, tk['likeStatus'])
                print(f"rating matched song: {query} as {tk['likeStatus']}")


    pl_res = yt.create_playlist(title=new_pl_name, video_ids=new_pl_ids, description=f"based on {tk['album']['name']} cd collection")
    print(f"{pl_name} original # entires: {len(pl['tracks'])}, queried #: {len(new_pl_ids)}, new # entries {len(pl_res)}")

jukebox_1960s PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs
rating matched song: Del Shannon Keep Searchin' as LIKE
rating matched song: Connie Francis La Vie En Rose as LIKE
rating matched song: The Ventures Cruel Sea as LIKE
rating matched song: Frank Sinatra Hello Dolly! as LIKE
rating matched song: Brian Hyland Sealed with a Kiss as LIKE
jukebox_1960s original # entires: 695, queried #: 718, new # entries 34
jukebox_1950s PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj
rating matched song: Pee Wee King & His Cowboys Slow Poke as LIKE
rating matched song: Bill Haley & His Comets Shake Rattle & Roll as LIKE
rating matched song: Kay Starr The Prisoner's Song as LIKE
rating matched song: Tennessee Ernie Ford Sixteen Tons as LIKE
rating matched song: Bill Hayes The Ballad of Davy Crockett as LIKE
rating matched song: Roger Williams Autumn Leaves as LIKE
rating matched song: Bill Haley & The Comets Burn That Candle as LIKE
rating matched song: Gene Vincent & Blue Caps Be Bop A Lula as LIKE
rating matched song: 